# TP 4 - Performance HDFS : petits fichiers, formats et partitionnement

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syhard87/tp_hadloop/blob/claude/tp-hadoop-avances-partie-a-mwgfq7/notebooks/TP4_Performance_HDFS.ipynb)

Source : `sources/TP_Hadoop_avances_avec_TP_ecosysteme.pdf`, pages 15 et 16. Guide pour le passage au tableau : `docs/TP4_Guide_tableau.md`.

**Contexte** : une application dépose **un fichier CSV par minute**. Après plusieurs mois, HDFS contient un très grand nombre
de petits fichiers. Les requêtes deviennent coûteuses et la charge du NameNode augmente.

**Mode d'emploi** : se connecter (coche verte en haut à droite), puis exécuter les cellules **dans l'ordre**.
Les cellules marquées ⏱ sont longues (plusieurs minutes).

> **Message `Connection refused` sur `localhost:9000`** : HDFS ne tourne pas. Relancer la **cellule 2** et attendre `✅ Hadoop prêt`.

## 1. Récupérer le dépôt

In [ ]:
!rm -rf /content/sample_data /content/tp_hadloop
!git clone -q --depth 1 -b claude/tp-hadoop-avances-partie-a-mwgfq7 https://github.com/Syhard87/tp_hadloop.git /content/tp_hadloop
!ls -l /content/tp_hadloop/tp4

## 2. Installer et démarrer Hadoop (HDFS + YARN), puis les bibliothèques de formats

In [ ]:
import os
HADOOP_HOME = "/content/hadoop-3.5.0"
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
if f"{HADOOP_HOME}/bin" not in os.environ["PATH"]:
    os.environ["PATH"] = f"{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:" + os.environ["PATH"]
os.makedirs("/content/preuves_tp4", exist_ok=True)

!bash /content/tp_hadloop/scripts/setup_hadoop_colab.sh
!pip install -q fastavro pyarrow

# Arrête « Tout exécuter » ici si Hadoop n'a pas démarré (sinon les cellules suivantes échoueraient en silence)
import subprocess
if "OFF" not in subprocess.run(["hdfs", "dfsadmin", "-safemode", "get"], capture_output=True, text=True).stdout:
    raise RuntimeError("Hadoop n'est pas prêt : lire le message ❌ ci-dessus, puis relancer cette cellule")

## Mission A - Reproduire le problème
On simule **200 minutes** de dépôts : 200 fichiers d'**une seule ligne** (22 octets chacun), envoyés dans HDFS.
On chronomètre le `put` avec `time`.

In [ ]:
%%bash
set -e
rm -rf ~/tp4-smallfiles
hdfs dfs -rm -r -f /tp4 > /dev/null 2>&1 || true

mkdir -p ~/tp4-smallfiles/source
for i in $(seq -w 1 200); do
 echo "2026-10-11;FR;P01;$i" > ~/tp4-smallfiles/source/event_$i.csv
done
echo "Fichiers créés en local : $(ls ~/tp4-smallfiles/source | wc -l) ; exemple : $(cat ~/tp4-smallfiles/source/event_001.csv)"

BLOCS_AVANT=$(find /content/hdfs/data -name 'blk_*' ! -name '*.meta' | wc -l)
hdfs dfs -mkdir -p /tp4/raw/events
echo; echo "=== put de 200 petits fichiers (chronométré)"
time hdfs dfs -put ~/tp4-smallfiles/source/*.csv /tp4/raw/events/
BLOCS_APRES=$(find /content/hdfs/data -name 'blk_*' ! -name '*.meta' | wc -l)

echo; echo "=== hdfs dfs -count -q -h  (avec -v pour afficher les en-têtes de colonnes)"
hdfs dfs -count -q -h -v /tp4/raw/events | tee /content/preuves_tp4/A_count.txt
echo; echo "=== hdfs dfs -du -h | head"
hdfs dfs -du -h /tp4/raw/events | head | tee /content/preuves_tp4/A_du.txt
echo; echo "=== Bilan fsck"
hdfs fsck /tp4/raw/events 2>/dev/null | grep -E "Total size|Total files|Total blocks" | tee /content/preuves_tp4/A_fsck.txt
echo; echo "=== Sur le disque du DataNode : $((BLOCS_APRES - BLOCS_AVANT)) nouveaux fichiers de blocs (+ autant de fichiers .meta de sommes de contrôle)"
echo "=== Sur le disque local : $(du -sh ~/tp4-smallfiles/source | cut -f1) occupés pour $(cat ~/tp4-smallfiles/source/*.csv | wc -c) octets de données"

**Lecture de `-count -q`** :
- `QUOTA` / `REM_QUOTA` : quota de **noms** (fichiers + répertoires) et ce qu'il en reste. `none` / `inf` = pas de quota.
- `SPACE_QUOTA` / `REM_SPACE_QUOTA` : quota d'**espace** (octets, réplication comprise).
- `DIR_COUNT` = **1**, `FILE_COUNT` = **200**, `CONTENT_SIZE` = **4,3 K**.

### Calcul de la charge du NameNode
Ordre de grandeur couramment admis : chaque **objet** de l'espace de noms (fichier, répertoire, bloc) coûte environ
**150 octets de mémoire** (heap Java) dans le NameNode.

In [ ]:
OCTETS_PAR_OBJET = 150
def objets(fichiers, repertoires=1, blocs_par_fichier=1):
    return fichiers + repertoires + fichiers * blocs_par_fichier

def fmt(o):
    for u in ["o", "Ko", "Mo", "Go"]:
        if o < 1024: return f"{o:,.0f} {u}".replace(",", " ")
        o /= 1024
    return f"{o:.1f} To"

print(f"TP (200 fichiers)      : {objets(200):>9,} objets -> {fmt(objets(200) * OCTETS_PAR_OBJET)} de heap pour 4,3 Ko de données".replace(",", " "))
print(f"Après compactage       : {objets(1):>9,} objets -> {fmt(objets(1) * OCTETS_PAR_OBJET)}".replace(",", " "))
an = 365 * 24 * 60
print(f"1 fichier/min pendant 1 an          : {objets(an):>11,} objets -> {fmt(objets(an) * OCTETS_PAR_OBJET)} de heap".replace(",", " "))
print(f"... pour 100 applications           : {objets(an) * 100:>11,} objets -> {fmt(objets(an) * 100 * OCTETS_PAR_OBJET)} de heap".replace(",", " "))

### Réponses
**23. Compter les fichiers.** `FILE_COUNT` = **200** fichiers dans **1** répertoire, et `fsck` compte **200 blocs**
(`avg. block size 22 B`). Chaque fichier occupe son propre bloc, même s'il ne contient que 22 octets.

**24. Volume total et nombre d'objets HDFS.** Le volume total n'est que de **4 400 octets (4,3 Ko)**, soit
**0,003 %** d'un seul bloc de 128 Mo. Pourtant, il génère **401 objets** dans le NameNode (200 fichiers + 200 blocs
+ 1 répertoire), contre **3** pour la même donnée dans un seul fichier. Le ratio volume / objets est catastrophique :
la donnée tiendrait 30 000 fois dans un seul bloc. Côté disque, le DataNode crée en plus **400 fichiers locaux**
(200 `blk_…` + 200 `.meta`). Et le `put` a pris environ **55 secondes** pour 4,3 Ko : chaque fichier coûte des
allers-retours avec le NameNode (création, allocation du bloc, fermeture) et l'ouverture d'un pipeline d'écriture.

**25. Pourquoi ce cas surcharge les métadonnées.**
- Le NameNode garde **toute** l'arborescence et la carte des blocs **en mémoire**, pour répondre vite. Environ
  150 octets par objet, **quelle que soit la taille du fichier** : un fichier de 22 octets coûte autant qu'un fichier de 128 Mo.
- Les **DataNodes** envoient des rapports de blocs : 200 blocs à déclarer au lieu de 1.
- **À l'échelle** : 1 fichier par minute donne 525 600 fichiers par an, soit environ 1 million d'objets et environ
  150 Mo de heap **par application**. Avec 100 applications, il faut 15 Go de heap. Le NameNode est **unique** : sa
  mémoire limite la taille du cluster, et ses longues pauses de *garbage collector* ralentissent tout.
- **Les traitements souffrent aussi** : en MapReduce, 1 petit fichier = au moins **1 split = 1 tâche Map = 1 conteneur JVM**.
  Le démarrage d'une JVM (environ 1 s) coûte bien plus cher que le traitement de 22 octets (voir le bonus de la Mission B).

### Bonus A - Le quota de noms (colonnes `QUOTA` / `REM_QUOTA`)
Un administrateur peut **limiter le nombre d'objets** d'un répertoire, pour protéger le NameNode d'une application qui
produirait trop de fichiers. Le répertoire lui-même compte pour 1.

In [ ]:
%%bash
hdfs dfs -mkdir -p /tp4/quota
hdfs dfsadmin -setQuota 50 /tp4/quota
hdfs dfs -count -q -v /tp4/quota
echo; echo "=== Tentative de dépôt de 100 fichiers"
hdfs dfs -put ~/tp4-smallfiles/source/event_0*.csv /tp4/quota/ 2>&1 | grep -m1 "quota"
hdfs dfs -count -q -v /tp4/quota | tee /content/preuves_tp4/A_quota.txt
hdfs dfsadmin -clrQuota /tp4/quota && hdfs dfs -rm -r -f /tp4/quota > /dev/null

## Mission B - Compacter
On regroupe les 200 fichiers en **un seul** (`cat`), puis on le dépose dans la zone `curated`.

In [ ]:
%%bash
set -e
cat ~/tp4-smallfiles/source/*.csv > ~/tp4-smallfiles/events_compactes.csv
echo "Fichier compacté : $(wc -l < ~/tp4-smallfiles/events_compactes.csv) lignes, $(wc -c < ~/tp4-smallfiles/events_compactes.csv) octets"
hdfs dfs -mkdir -p /tp4/curated/events
echo; echo "=== put du fichier compacté (chronométré)"
time hdfs dfs -put -f ~/tp4-smallfiles/events_compactes.csv /tp4/curated/events/
echo
hdfs fsck /tp4/curated/events/events_compactes.csv -files -blocks -locations 2>/dev/null | sed '/^Erasure/,$d' | tee /content/preuves_tp4/B_fsck.txt
echo; echo "=== Avant / après"
hdfs dfs -count -v /tp4/raw/events /tp4/curated/events | tee /content/preuves_tp4/B_count.txt

In [ ]:
%%bash
echo "=== Variante : compacter directement depuis HDFS (sans garder la source locale)"
hdfs dfs -getmerge /tp4/raw/events /tmp/merge.csv
cmp /tmp/merge.csv ~/tp4-smallfiles/events_compactes.csv && echo "getmerge produit exactement le même fichier"
echo
echo "=== Contrôle d'intégrité avant de supprimer les petits fichiers (même nombre de lignes)"
echo "raw     : $(hdfs dfs -cat /tp4/raw/events/*.csv | wc -l) lignes"
echo "curated : $(hdfs dfs -cat /tp4/curated/events/events_compactes.csv | wc -l) lignes"

### Réponses
| | Avant (`/tp4/raw/events`) | Après (`/tp4/curated/events`) | Gain |
|---|---|---|---|
| Fichiers | **200** | **1** | ÷ 200 |
| Blocs | **200** (22 o en moyenne) | **1** (4 400 o) | ÷ 200 |
| Objets NameNode (fichiers + blocs + répertoire) | **401** | **3** | ÷ 134 |
| Volume de données | 4 400 o | 4 400 o | identique |
| Durée du `put` | **≈ 55 s** | **≈ 2 s** | ≈ 25× plus rapide |
| Tâches Map (bonus ci-dessous) | **200** | **2** | ÷ 100 |

**Gain pour le NameNode** : 134 fois moins d'objets en mémoire, 200 fois moins de blocs à suivre dans les rapports
des DataNodes, et beaucoup moins de requêtes (`open`, `getBlockLocations`) à chaque lecture.

**Gain pour les traitements** :
- un seul fichier à ouvrir au lieu de 200 ;
- des lectures **séquentielles** au lieu d'aléatoires ;
- **moins de tâches Map**, donc moins de JVM à démarrer.

Le contenu est **strictement identique** : même nombre de lignes, et `getmerge` donne le même fichier. On peut donc
supprimer les petits fichiers **après contrôle**.

### Bonus B - ⏱ Le coût réel pour MapReduce (environ 7 minutes)
Le **même** job de comptage de lignes est lancé 3 fois sur les **mêmes 200 lignes** :
1. sur le fichier compacté ;
2. sur les 200 petits fichiers avec `CombineTextInputFormat`, qui regroupe plusieurs petits fichiers dans un même split ;
3. sur les 200 petits fichiers, tels quels.

In [ ]:
%%bash
J=$(find "$HADOOP_HOME/share/hadoop/tools/lib" -name 'hadoop-streaming-*.jar' | head -1)
lancer() {  # $1 = nom, $2 = entrée, $3 = options supplémentaires
  hdfs dfs -rm -r -f /tp4/job_$1 > /dev/null 2>&1
  local debut=$(date +%s)
  hadoop jar $J -D mapreduce.job.name=TP4_$1 -D mapreduce.job.reduces=1 $3 \
    -mapper cat -reducer "wc -l" -input $2 -output /tp4/job_$1 > /tmp/job_$1.log 2>&1
  printf "%-28s durée=%4ss  %-22s  lignes comptées=%s\n" "$1" "$(( $(date +%s) - debut ))" \
    "$(grep -o 'Launched map tasks=[0-9]*' /tmp/job_$1.log)" "$(hdfs dfs -cat /tp4/job_$1/part-00000 | tr -d '\t ')"
}
lancer 1_fichier_compacte /tp4/curated/events
lancer 2_petits_CombineInput /tp4/raw/events "-inputformat org.apache.hadoop.mapred.lib.CombineTextInputFormat"
lancer 3_petits_fichiers /tp4/raw/events

**Résultat obtenu lors de notre test** : 28 s et 2 tâches Map pour le fichier compacté ; 20 s et 1 tâche Map avec
`CombineTextInputFormat` ; **343 s et 200 tâches Map** pour les petits fichiers, soit **12 fois plus lent pour compter
200 lignes**. Le temps CPU total des Maps est passé de 4 s à 187 s : presque tout ce temps sert à **démarrer des JVM**,
pas à traiter des données.

## Mission C - Concevoir un partitionnement
Les événements sont consultés **principalement par date et pays**.

**26. Clés de partitionnement proposées**
```
/data/curated/events/year=AAAA/month=MM/day=JJ/country=XX/part-00000.parquet
```
- `year`, `month` et `day` viennent de la **date de l'événement** (date métier, à ne pas confondre avec `ingestion_date`
  dans `raw`, qui est la date de réception).
- `country` est le code pays, en majuscules et normalisé (`FR`, `BE`…).
- La forme `cle=valeur` (style **Hive**) est comprise par Hive, Spark, Impala et Trino : la clé devient une
  **colonne virtuelle** sur laquelle filtrer.

**27. Justifier l'ordre des partitions**
- **Du plus général au plus fin, en suivant les filtres les plus fréquents.** Presque toutes les requêtes filtrent sur
  une **période** : la date vient donc en premier. Une plage de dates (« octobre », « la semaine dernière ») correspond
  alors à un **sous-arbre contigu**.
- `year/month/day` donne une hiérarchie naturelle : un mois entier se lit avec `year=2026/month=10/*`.
- `country` vient **en dernier**, car il a peu de valeurs (une dizaine) et sert à affiner à l'intérieur d'une journée.
- **Cycle de vie** : la purge et l'archivage se font par date. Supprimer un vieux mois revient à supprimer **un seul
  répertoire**. Avec `country` en premier, il faudrait parcourir chaque pays.
- **Écriture** : les données arrivent dans l'ordre du temps, donc un traitement quotidien n'écrit que dans la
  partition `day=` du jour.

**28. Risque des partitions trop fines** : chaque combinaison de valeurs crée un répertoire et au moins un fichier.
Des partitions trop fines **recréent le problème des petits fichiers** de la Mission A : surcharge du NameNode, trop
de tâches, des métadonnées plus lourdes que les données. Le planificateur de requêtes doit aussi lister des milliers
de répertoires avant même de lire une donnée. La démonstration ci-dessous le chiffre.

In [ ]:
!cd /content/tp_hadloop/tp4 && python3 partitionner.py /content/tp4/events_partitionnes | tee /content/preuves_tp4/C_partitionnement.txt

**Lecture du résultat** :
- L'**élagage de partitions** (*partition pruning*) fonctionne : « un jour + un pays » n'ouvre qu'**1 fichier sur 240**.
  Une requête qui filtre sur une colonne **qui n'est pas** une clé de partition (`produit`) doit tout ouvrir.
- **Mais** avec ce volume (environ 14 000 événements par jour), `year/month/day/country` produit des fichiers de
  **38 Ko** en moyenne. C'est 3 000 fois plus petit qu'un bloc : **trop fin**. Ajouter l'heure, ou garder un fichier par
  minute, serait pire.
- **Conclusion nuancée** : la forme proposée par l'énoncé convient si **chaque partition jour × pays pèse au moins
  plusieurs centaines de Mo**. Sinon, on partitionne par `year/month/day`, voire `year/month`, et on garde `country`
  comme **colonne triée** dans des fichiers Parquet. Les statistiques min/max de Parquet permettent alors de sauter les
  blocs de lignes inutiles, sans créer de répertoires.

On dépose l'arborescence dans HDFS pour la visualiser (environ 1 minute pour 240 fichiers, ce qui illustre aussi le coût des petits fichiers) :

In [ ]:
%%bash
hdfs dfs -rm -r -f /data/curated/events > /dev/null 2>&1
hdfs dfs -mkdir -p /data/curated
time hdfs dfs -put /content/tp4/events_partitionnes /data/curated/events
echo
hdfs dfs -ls -R /data/curated/events | grep "day=15" | head -n 10
echo "..."
echo; echo "=== Bilan"
hdfs dfs -count -v /data/curated/events | tee /content/preuves_tp4/C_count_hdfs.txt
echo "Partition « 15 octobre, FR » :"; hdfs dfs -ls /data/curated/events/year=2026/month=10/day=15/country=FR

### 29. Comparer CSV gzip, Avro, Parquet et ORC pour ce cas analytique
**Mesure réelle** : 30 jours d'événements (432 000 lignes, 8 colonnes), écrits dans chaque format, puis une requête
analytique typique : *chiffre d'affaires par pays pour un jour donné*. Elle n'a besoin que de 3 colonnes sur 8 et
d'1 jour sur 30.

In [ ]:
!cd /content/tp_hadloop/tp4 && python3 comparer_formats.py /content/tp4/formats | tee /content/preuves_tp4/C_formats.txt

**Le piège de gzip : il n'est pas découpable.** On dépose le CSV et le CSV gzip avec des blocs de **1 Mo**, pour voir
l'effet à petite échelle, puis on compte les tâches Map lancées par Hadoop (⏱ environ 1 minute 30).

In [ ]:
%%bash
J=$(find "$HADOOP_HOME/share/hadoop/tools/lib" -name 'hadoop-streaming-*.jar' | head -1)
hdfs dfs -mkdir -p /tp4/formats/csv /tp4/formats/gz
hdfs dfs -D dfs.blocksize=1048576 -put -f /content/tp4/formats/events.csv /tp4/formats/csv/
hdfs dfs -D dfs.blocksize=1048576 -put -f /content/tp4/formats/events.csv.gz /tp4/formats/gz/
hdfs fsck /tp4/formats -files 2>/dev/null | grep "bytes, replicated"
echo
for f in csv gz; do
  hdfs dfs -rm -r -f /tp4/job_$f > /dev/null 2>&1
  hadoop jar $J -D mapreduce.job.reduces=1 -mapper "wc -l" -reducer "awk '{s+=\$1} END {print s}'" \
    -input /tp4/formats/$f -output /tp4/job_$f > /tmp/job_$f.log 2>&1
  echo "$f : $(grep -o 'number of splits:[0-9]*' /tmp/job_$f.log) -> lignes comptées : $(hdfs dfs -cat /tp4/job_$f/part-00000)"
done | tee /content/preuves_tp4/C_gzip_splits.txt

| Critère | CSV gzip | Avro | Parquet | ORC |
|---|---|---|---|---|
| Organisation | Texte, **par ligne** | Binaire, **par ligne** | Binaire, **par colonne** | Binaire, **par colonne** |
| Schéma | Aucun (tout est texte) | **Intégré**, avec évolution de schéma | Intégré | Intégré |
| Découpable par Hadoop | **Non** (1 fichier = 1 tâche Map) | Oui (blocs de synchronisation) | Oui (row groups) | Oui (stripes) |
| Lecture de 3 colonnes sur 8 | Lit **tout** | Lit **tout** | Lit **seulement 3 colonnes** | Lit **seulement 3 colonnes** |
| Filtre `date = …` | Après lecture complète | Après lecture complète | **Statistiques min/max** : saute des blocs | **Index et statistiques** : saute des stripes |
| Taille mesurée | 5,1 Mo (20 %) | 5,2 Mo (20 %) | 6,0 Mo (23 %) | **3,2 Mo (12 %)** |
| Requête mesurée | 0,17 s | 2,0 s (*) | **0,03 s** | 0,06 s |
| Usage idéal | Échange, archivage simple | **Ingestion** et flux (Kafka), écritures ligne par ligne | **Analyse** (Spark, Trino, Hive), standard de fait | **Analyse** dans l'écosystème Hive |

(*) Avro est lu ici par une bibliothèque Python ligne par ligne, ce qui exagère sa lenteur. Le fond reste vrai : un
format ligne doit tout décoder.

**Recommandation pour ce cas analytique** : **Parquet** (ou ORC si l'entreprise est centrée sur Hive). Les requêtes ne
lisent que quelques colonnes et filtrent par date et pays : un format **colonne** avec statistiques est fait pour ça.
**Avro** reste pertinent **en amont**, pour l'ingestion minute par minute. **CSV gzip** est à éviter pour l'analyse :
non découpable, sans schéma, il oblige à tout décompresser.

## Mission D - Décision argumentée (15 lignes maximum)

1. **Fichier par minute** : seulement en **zone d'atterrissage temporaire**, jamais comme stockage durable (525 600 fichiers par an et par flux).
2. **Compaction planifiée** à la clôture d'une période (chaque nuit pour la veille), et **sur seuil** (trop de fichiers ou taille moyenne trop faible).
3. **Compaction sûre** : écriture dans un dossier temporaire, contrôle des lignes, `mv` atomique et `_SUCCESS`, puis suppression des petits fichiers.
4. **Format** : **Parquet** compressé (snappy ou zstd) pour l'analyse, ORC si l'environnement est Hive ; Avro pour l'ingestion ; pas de CSV gzip.
5. **Partitionnement** : `year/month/day`, et `country` seulement si chaque partition jour × pays reste volumineuse. Sinon, une colonne triée.
6. **Taille cible** : pas de règle universelle. Des fichiers de l'ordre d'un ou plusieurs blocs (centaines de Mo), qui amortissent ouverture et tâche, mais en nombre suffisant pour occuper tous les cœurs. On ajuste au volume : si une journée pèse 300 Ko, on regroupe au mois.
7. **Contrôles automatisés** : nombre de fichiers et taille moyenne par partition (`-count`), objets du NameNode (`FilesTotal`, `BlocksTotal`), quotas de noms, égalité des lignes avant et après, `_SUCCESS`, `fsck`, rétention.

## Sauvegarder les preuves dans l'espace partagé (Drive)

In [ ]:
from google.colab import drive
import shutil
drive.mount('/content/drive')
dest = "/content/drive/MyDrive/TP_Hadoop_partage/preuves_tp4"
shutil.copytree("/content/preuves_tp4", dest, dirs_exist_ok=True)
print("Preuves copiées dans", dest, ":", sorted(os.listdir(dest)))